# Image Captioning with CNN-LSTM and Attention

Complete implementation of an image captioning system using:
- ResNet-50 CNN Encoder
- LSTM Decoder with Bahdanau Attention
- Flickr8k Dataset

**GPU Optimized for CUDA/MPS**

## 1. Install Required Packages

In [1]:
# Install required packages if needed
!pip install torch torchvision Pillow numpy matplotlib nltk tqdm jupyter ipykernel

  Using cached fqdn-1.5.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached webcolors-25.10.0-py3-none-any.whl.metadata (2.2 kB)
Using cached webcolors-25.10.0-py3-none-any.whl (14 kB)
Using cached fqdn-1.5.1-py3-none-any.whl (9.1 kB)
Using cached isoduration-20.11.0-py3-none-any.whl (11 kB)
Using cached uri_template-1.3.0-py3-none-any.whl (11 kB)


## 2. Import Libraries

In [4]:
import os
import re
import random
import urllib.request
import zipfile
from collections import Counter

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tqdm import tqdm

# Check PyTorch version and GPU availability
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
print(f"MPS Available: {torch.backends.mps.is_available()}")

PyTorch Version: 2.7.1+cu118
CUDA Available: True
CUDA Device: NVIDIA GeForce RTX 4080 Laptop GPU
Number of GPUs: 1
MPS Available: False


## 3. Configuration Class

In [2]:
class Config:
    """Hyperparameters and paths"""
    # Data paths
    data_dir = "data/Flickr8k"
    images_dir = os.path.join(data_dir, "Images")
    captions_file = os.path.join(data_dir, "captions.txt")

    # Training split ratios
    train_ratio = 0.8
    val_ratio = 0.1  # test = 0.1

    # Vocabulary settings
    min_freq = 5
    max_caption_len = 40

    # Model architecture
    embed_dim = 256
    encoder_dim = 512
    decoder_dim = 512
    attention_dim = 256

    # Training hyperparameters
    batch_size = 64
    num_workers = 4
    lr = 1e-4
    num_epochs = 10
    grad_clip = 5.0
    
    # Device - GPU optimized
    device = (
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )

    # Special tokens
    pad_token = "<pad>"
    start_token = "<start>"
    end_token = "<end>"
    unk_token = "<unk>"

cfg = Config()
print(f"Using device: {cfg.device}")

Using device: cuda


## 4. Download and Prepare Dataset

In [3]:
def download_flickr8k():
    """Download and extract Flickr8k dataset"""
    print("Downloading Flickr8k dataset...")
    
    # Create directories
    os.makedirs('data/Flickr8k', exist_ok=True)
    
    # Download (from alternative sources)
    dataset_url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip"
    captions_url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"
    
    print("Downloading images...")
    urllib.request.urlretrieve(dataset_url, "Flickr8k_Dataset.zip")
    
    print("Downloading captions...")
    urllib.request.urlretrieve(captions_url, "Flickr8k_text.zip")
    
    print("Extracting...")
    with zipfile.ZipFile("Flickr8k_Dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("data/Flickr8k")
    
    with zipfile.ZipFile("Flickr8k_text.zip", 'r') as zip_ref:
        zip_ref.extractall("data/Flickr8k")
    
    # Rename captions file
    if os.path.exists("data/Flickr8k/Flickr8k.token.txt"):
        os.rename("data/Flickr8k/Flickr8k.token.txt", "data/Flickr8k/captions.txt")
    
    # Clean up zip files
    os.remove("Flickr8k_Dataset.zip")
    os.remove("Flickr8k_text.zip")
    
    print("✓ Dataset ready!")

# Download dataset if not already present
if not os.path.exists(cfg.captions_file):
    download_flickr8k()
else:
    print("Dataset already exists!")

Dataset already exists!


## 5. Utility Functions

In [ ]:
def clean_caption(caption: str) -> str:
    """Clean and normalize caption text"""
    caption = caption.lower().strip()
    caption = re.sub(r"[^a-z0-9,.!?']", " ", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption


def tokenize(caption: str):
    """Simple whitespace tokenization"""
    return caption.split()

## 6. Vocabulary Class

In [ ]:
class Vocabulary:
    """Build and manage vocabulary"""
    def __init__(self, min_freq=5):
        self.min_freq = min_freq
        self.freqs = Counter()
        self.stoi = {}
        self.itos = []

        self.pad_token = cfg.pad_token
        self.start_token = cfg.start_token
        self.end_token = cfg.end_token
        self.unk_token = cfg.unk_token

    def build(self, all_captions):
        """Build vocabulary from captions"""
        # Count word frequencies
        for cap in all_captions:
            tokens = tokenize(clean_caption(cap))
            self.freqs.update(tokens)

        # Add special tokens first
        self.itos = [
            self.pad_token,
            self.start_token,
            self.end_token,
            self.unk_token,
        ]
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}

        # Add words that meet frequency threshold
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def numericalize(self, caption: str):
        """Convert caption to token IDs"""
        tokens = tokenize(clean_caption(caption))
        return [self.stoi.get(tok, self.stoi[self.unk_token]) for tok in tokens]

    def __len__(self):
        return len(self.itos)

## 7. Data Loading Functions

In [ ]:
def load_captions(captions_file):
    """
    Load captions from file
    
    Returns:
        image2caps: dict mapping image_name to list of captions
        all_pairs: list of (image_name, caption) tuples
    """
    image2caps = {}
    all_pairs = []

    with open(captions_file, "r") as f:
        for line in f:
            if not line.strip():
                continue

            # Split on first comma
            parts = line.strip().split(",", 1)
            if len(parts) != 2:
                continue
                
            img_id, caption = parts
            img_name = img_id.strip()
            caption = caption.strip()

            # Skip header or non-image rows
            if not img_name.lower().endswith(".jpg"):
                continue

            image2caps.setdefault(img_name, []).append(caption)
            all_pairs.append((img_name, caption))

    return image2caps, all_pairs


def train_val_test_split(all_pairs, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Split data into train/val/test sets"""
    random.seed(seed)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)
    
    train_data = all_pairs[:n_train]
    val_data = all_pairs[n_train:n_train + n_val]
    test_data = all_pairs[n_train + n_val:]
    
    return train_data, val_data, test_data

## 8. Dataset Class

In [ ]:
class FlickrCaptionDataset(Dataset):
    """Flickr8k dataset loader"""
    def __init__(self, image_caption_pairs, images_dir, vocab: Vocabulary,
                 transform=None, max_len=40):
        self.data = image_caption_pairs
        self.images_dir = images_dir
        self.vocab = vocab
        self.transform = transform
        self.max_len = max_len

        self.start_idx = vocab.stoi[cfg.start_token]
        self.end_idx = vocab.stoi[cfg.end_token]
        self.pad_idx = vocab.stoi[cfg.pad_token]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name, caption = self.data[idx]
        img_path = os.path.join(self.images_dir, img_name)

        # Load image
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Encode caption: <start> tokens <end>
        token_ids = self.vocab.numericalize(caption)
        token_ids = token_ids[: self.max_len - 2]
        caption_ids = [self.start_idx] + token_ids + [self.end_idx]
        length = len(caption_ids)

        return image, torch.tensor(caption_ids, dtype=torch.long), length

## 9. Custom Collate Function

In [ ]:
class CaptionCollate:
    """Custom collate function for variable-length captions"""
    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        images, captions, lengths = zip(*batch)
        images = torch.stack(images, dim=0)

        # Pad captions to max length in batch
        max_len = max(lengths)
        padded_captions = torch.full(
            (len(captions), max_len),
            fill_value=self.pad_idx,
            dtype=torch.long,
        )

        for i, cap in enumerate(captions):
            end = cap.shape[0]
            padded_captions[i, :end] = cap

        lengths = torch.tensor(lengths, dtype=torch.long)

        return images, padded_captions, lengths

## 10. CNN Encoder

In [ ]:
class EncoderCNN(nn.Module):
    """ResNet-50 based encoder"""
    def __init__(self, encoded_image_size=14, encoder_dim=512):
        super().__init__()
        self.enc_image_size = encoded_image_size

        # Load pretrained ResNet-50
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        
        # Remove final layers (keep conv features)
        modules = list(resnet.children())[:-2]
        self.cnn = nn.Sequential(*modules)

        # Adaptive pooling to fixed size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))

        # Project from 2048 to encoder_dim
        self.conv_project = nn.Conv2d(2048, encoder_dim, kernel_size=1, stride=1)

        self.fine_tune(False)

    def forward(self, images):
        """
        Args:
            images: (B, 3, H, W)
        Returns:
            features: (B, L, encoder_dim) where L = enc_size^2
        """
        features = self.cnn(images)              # (B, 2048, H', W')
        features = self.adaptive_pool(features)   # (B, 2048, enc_size, enc_size)
        features = self.conv_project(features)    # (B, encoder_dim, enc_size, enc_size)
        
        # Reshape to sequence
        B, D, H, W = features.size()
        features = features.permute(0, 2, 3, 1).view(B, -1, D)  # (B, L, D)
        
        return features

    def fine_tune(self, fine_tune=True):
        """Freeze/unfreeze encoder layers"""
        for p in self.cnn.parameters():
            p.requires_grad = False
        # Optionally unfreeze later layers
        for c in list(self.cnn.children())[-2:]:
            for p in c.parameters():
                p.requires_grad = fine_tune

## 11. Bahdanau Attention Mechanism

In [ ]:
class BahdanauAttention(nn.Module):
    """Additive attention mechanism"""
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        """
        Args:
            encoder_out: (B, L, encoder_dim)
            decoder_hidden: (B, decoder_dim)
        Returns:
            context: (B, encoder_dim)
            alpha: (B, L) attention weights
        """
        att1 = self.encoder_att(encoder_out)  # (B, L, att_dim)
        att2 = self.decoder_att(decoder_hidden).unsqueeze(1)  # (B, 1, att_dim)
        att = self.full_att(self.relu(att1 + att2)).squeeze(2)  # (B, L)
        alpha = self.softmax(att)  # (B, L)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)  # (B, encoder_dim)
        return context, alpha

## 12. LSTM Decoder with Attention

In [ ]:
class DecoderWithAttention(nn.Module):
    """LSTM decoder with attention"""
    def __init__(self, vocab_size, embed_dim, encoder_dim, decoder_dim, attention_dim, pad_idx):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx

        self.attention = BahdanauAttention(encoder_dim, decoder_dim, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        
        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)
        
        self.lstm_cell = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)
        self.sigmoid = nn.Sigmoid()
        
        self.fc = nn.Linear(decoder_dim, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def init_hidden_state(self, encoder_out):
        """Initialize LSTM state from encoder output"""
        mean_encoder = encoder_out.mean(dim=1)  # (B, encoder_dim)
        h = self.init_h(mean_encoder)
        c = self.init_c(mean_encoder)
        return h, c

    def forward(self, encoder_out, captions, lengths):
        """
        Args:
            encoder_out: (B, L, encoder_dim)
            captions: (B, max_len)
            lengths: (B,)
        Returns:
            predictions: (B, max_len-1, vocab_size)
            captions_sorted: sorted captions
            decode_lengths: actual decode lengths
            alphas: attention weights
            sort_idx: sorting indices
        """
        batch_size = encoder_out.size(0)
        L = encoder_out.size(1)

        # Sort by length
        lengths_sorted, sort_idx = lengths.sort(dim=0, descending=True)
        encoder_out = encoder_out[sort_idx]
        captions = captions[sort_idx]

        # Embed captions
        embeddings = self.embedding(captions)  # (B, max_len, embed_dim)

        # Initialize LSTM
        h, c = self.init_hidden_state(encoder_out)

        # Decode lengths
        decode_lengths = (lengths_sorted - 1).tolist()
        max_decode_len = max(decode_lengths)

        # Storage
        predictions = torch.zeros(batch_size, max_decode_len, self.vocab_size).to(encoder_out.device)
        alphas = torch.zeros(batch_size, max_decode_len, L).to(encoder_out.device)

        # Decode step by step
        for t in range(max_decode_len):
            batch_t = sum([l > t for l in decode_lengths])
            
            # Attention
            context, alpha = self.attention(encoder_out[:batch_t], h[:batch_t])
            gate = self.sigmoid(self.f_beta(h[:batch_t]))
            context = gate * context

            # LSTM step
            lstm_input = torch.cat([embeddings[:batch_t, t, :], context], dim=1)
            h_new, c_new = self.lstm_cell(lstm_input, (h[:batch_t], c[:batch_t]))

            # Update states
            h = torch.cat([h_new, h[batch_t:]], dim=0)
            c = torch.cat([c_new, c[batch_t:]], dim=0)

            # Predict
            preds = self.fc(self.dropout(h_new))
            predictions[:batch_t, t, :] = preds
            alphas[:batch_t, t, :] = alpha

        return predictions, captions, decode_lengths, alphas, sort_idx

## 13. Create Data Loaders

In [ ]:
def create_dataloaders():
    """Create train/val/test dataloaders"""
    # Load captions
    image2caps, all_pairs = load_captions(cfg.captions_file)
    train_pairs, val_pairs, test_pairs = train_val_test_split(
        all_pairs, cfg.train_ratio, cfg.val_ratio
    )

    # Build vocabulary
    vocab = Vocabulary(min_freq=cfg.min_freq)
    vocab.build([cap for _, cap in train_pairs])
    cfg.pad_token_id = vocab.stoi[cfg.pad_token]

    print(f"Vocabulary size: {len(vocab)}")
    print(f"Training samples: {len(train_pairs)}")
    print(f"Validation samples: {len(val_pairs)}")
    print(f"Test samples: {len(test_pairs)}")

    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Datasets
    train_dataset = FlickrCaptionDataset(
        train_pairs, cfg.images_dir, vocab,
        transform=train_transform, max_len=cfg.max_caption_len
    )
    val_dataset = FlickrCaptionDataset(
        val_pairs, cfg.images_dir, vocab,
        transform=val_transform, max_len=cfg.max_caption_len
    )
    test_dataset = FlickrCaptionDataset(
        test_pairs, cfg.images_dir, vocab,
        transform=val_transform, max_len=cfg.max_caption_len
    )

    collate = CaptionCollate(pad_idx=cfg.pad_token_id)

    # Dataloaders - Use pin_memory for GPU
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, collate_fn=collate, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, collate_fn=collate, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, collate_fn=collate, pin_memory=True
    )

    return train_loader, val_loader, test_loader, vocab, image2caps, test_pairs

# Create dataloaders
train_loader, val_loader, test_loader, vocab, image2caps, test_pairs = create_dataloaders()

## 14. Training Functions

In [ ]:
def train_one_epoch(encoder, decoder, criterion, optimizer, train_loader, epoch):
    """Train for one epoch"""
    encoder.train()
    decoder.train()

    total_loss = 0.0

    for images, captions, lengths in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        images = images.to(cfg.device, non_blocking=True)
        captions = captions.to(cfg.device, non_blocking=True)
        lengths = lengths.to(cfg.device, non_blocking=True)

        optimizer.zero_grad()

        encoder_out = encoder(images)
        scores, caps_sorted, decode_lengths, alphas, sort_idx = decoder(
            encoder_out, captions, lengths
        )

        # Targets: next word after each position
        targets = caps_sorted[:, 1:]

        # Pack predictions and targets
        scores_packed = []
        targets_packed = []
        for i, l in enumerate(decode_lengths):
            scores_packed.append(scores[i, :l, :])
            targets_packed.append(targets[i, :l])

        scores_packed = torch.cat(scores_packed, dim=0)
        targets_packed = torch.cat(targets_packed, dim=0)

        loss = criterion(scores_packed, targets_packed)

        # Attention regularization (doubly stochastic)
        alphas_reg = 1.0 * ((1.0 - alphas.sum(dim=1)) ** 2).mean()
        loss = loss + alphas_reg

        loss.backward()
        nn.utils.clip_grad_norm_(decoder.parameters(), cfg.grad_clip)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
@torch.no_grad()
def validate(encoder, decoder, criterion, val_loader, epoch):
    """Validate the model"""
    encoder.eval()
    decoder.eval()

    total_loss = 0.0

    for images, captions, lengths in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
        images = images.to(cfg.device, non_blocking=True)
        captions = captions.to(cfg.device, non_blocking=True)
        lengths = lengths.to(cfg.device, non_blocking=True)

        encoder_out = encoder(images)
        scores, caps_sorted, decode_lengths, alphas, sort_idx = decoder(
            encoder_out, captions, lengths
        )

        targets = caps_sorted[:, 1:]

        scores_packed = []
        targets_packed = []
        for i, l in enumerate(decode_lengths):
            scores_packed.append(scores[i, :l, :])
            targets_packed.append(targets[i, :l])

        scores_packed = torch.cat(scores_packed, dim=0)
        targets_packed = torch.cat(targets_packed, dim=0)

        loss = criterion(scores_packed, targets_packed)
        alphas_reg = 1.0 * ((1.0 - alphas.sum(dim=1)) ** 2).mean()
        loss = loss + alphas_reg

        total_loss += loss.item()

    return total_loss / len(val_loader)

## 15. Inference - Caption Generation

In [ ]:
@torch.no_grad()
def generate_caption(encoder, decoder, image_tensor, vocab, max_len=20):
    """Generate caption for a single image"""
    encoder.eval()
    decoder.eval()

    image_tensor = image_tensor.to(cfg.device)
    encoder_out = encoder(image_tensor)

    h, c = decoder.init_hidden_state(encoder_out)
    start_idx = vocab.stoi[cfg.start_token]
    end_idx = vocab.stoi[cfg.end_token]

    word_idx = start_idx
    caption_idxs = [start_idx]

    for _ in range(max_len):
        word = torch.tensor([word_idx], dtype=torch.long).to(cfg.device)
        embeddings = decoder.embedding(word)

        context, alpha = decoder.attention(encoder_out, h)
        gate = decoder.sigmoid(decoder.f_beta(h))
        context = gate * context

        lstm_input = torch.cat([embeddings, context], dim=1)
        h, c = decoder.lstm_cell(lstm_input, (h, c))
        
        preds = decoder.fc(h)
        _, next_word = preds.max(dim=1)

        word_idx = next_word.item()
        caption_idxs.append(word_idx)
        
        if word_idx == end_idx:
            break

    # Convert to words
    words = []
    for idx in caption_idxs:
        tok = vocab.itos[idx]
        if tok in {cfg.start_token, cfg.end_token, cfg.pad_token}:
            continue
        words.append(tok)

    return " ".join(words)

## 16. BLEU Evaluation

In [ ]:
@torch.no_grad()
def evaluate_bleu_on_test(encoder, decoder, vocab, image2caps, test_pairs):
    """Compute BLEU-4 on test set"""
    encoder.eval()
    decoder.eval()

    # Get unique test images
    test_images = sorted(set(img_name for img_name, _ in test_pairs))

    references = []
    hypotheses = []

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    print(f"\nEvaluating BLEU on {len(test_images)} test images...")

    for img_name in tqdm(test_images, desc="BLEU eval"):
        img_path = os.path.join(cfg.images_dir, img_name)

        # Load image
        pil_img = Image.open(img_path).convert("RGB")
        img_tensor = transform(pil_img).unsqueeze(0)

        # Generate caption
        hyp_str = generate_caption(encoder, decoder, img_tensor, vocab, max_len=20)
        hyp_tokens = hyp_str.split()
        hypotheses.append(hyp_tokens)

        # Get reference captions
        ref_caps = image2caps[img_name]
        ref_tokens = []
        for cap in ref_caps:
            toks = tokenize(clean_caption(cap))
            ref_tokens.append(toks)
        references.append(ref_tokens)

    # BLEU-4 with smoothing
    smoothie = SmoothingFunction().method1
    bleu4 = corpus_bleu(references, hypotheses, smoothing_function=smoothie)
    
    print(f"\n{'='*60}")
    print(f"Test BLEU-4 Score: {bleu4:.4f}")
    print('='*60)
    
    return bleu4

## 17. Initialize Models and Optimizer

In [ ]:
print("="*60)
print("Image Captioning Training")
print("="*60)
print(f"Device: {cfg.device}")
print(f"Batch size: {cfg.batch_size}")
print(f"Epochs: {cfg.num_epochs}")
print(f"Learning rate: {cfg.lr}")
print("="*60)

# Build models
encoder = EncoderCNN(encoder_dim=cfg.encoder_dim).to(cfg.device)
decoder = DecoderWithAttention(
    vocab_size=len(vocab),
    embed_dim=cfg.embed_dim,
    encoder_dim=cfg.encoder_dim,
    decoder_dim=cfg.decoder_dim,
    attention_dim=cfg.attention_dim,
    pad_idx=vocab.stoi[cfg.pad_token],
).to(cfg.device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi[cfg.pad_token])
params = list(decoder.parameters()) + list(
    filter(lambda p: p.requires_grad, encoder.parameters())
)
optimizer = optim.Adam(params, lr=cfg.lr)

print(f"\n✓ Models initialized")
print(f"  Encoder parameters: {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}")
print(f"  Decoder parameters: {sum(p.numel() for p in decoder.parameters()):,}")

## 18. Main Training Loop

In [ ]:
best_val_loss = float("inf")
train_losses = []
val_losses = []

# Training loop
for epoch in range(1, cfg.num_epochs + 1):
    train_loss = train_one_epoch(
        encoder, decoder, criterion, optimizer, train_loader, epoch
    )
    val_loss = validate(encoder, decoder, criterion, val_loader, epoch)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"\nEpoch {epoch}/{cfg.num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        
        checkpoint = {
            "encoder": encoder.state_dict(),
            "decoder": decoder.state_dict(),
            "vocab": {
                "itos": vocab.itos,
                "stoi": vocab.stoi,
            },
            "config": {
                "encoder_dim": cfg.encoder_dim,
                "decoder_dim": cfg.decoder_dim,
                "embed_dim": cfg.embed_dim,
                "attention_dim": cfg.attention_dim,
            },
        }
        
        torch.save(checkpoint, "best_model.pth")
        print("  ✓ Saved best model")

print("\n" + "="*60)
print("Training complete!")
print("="*60)

## 19. Plot Training Curves

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss vs Epoch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training curves saved to training_curves.png")

## 20. Evaluate on Test Set (BLEU-4)

In [ ]:
# Final evaluation on test set
print("\n" + "="*60)
print("Evaluating on test set...")
print("="*60)

bleu4_score = evaluate_bleu_on_test(encoder, decoder, vocab, image2caps, test_pairs)

print(f"\n✓ Evaluation finished!")
print(f"Best model saved to: best_model.pth")

## 21. Visualize Sample Predictions

In [ ]:
# Visualize some sample predictions
def visualize_predictions(num_samples=5):
    """Visualize sample predictions with ground truth captions"""
    encoder.eval()
    decoder.eval()
    
    # Get random test images
    test_images = sorted(set(img_name for img_name, _ in test_pairs))
    sample_images = random.sample(test_images, min(num_samples, len(test_images)))
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    # Create figure
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = [axes]
    
    for idx, (ax, img_name) in enumerate(zip(axes, sample_images)):
        img_path = os.path.join(cfg.images_dir, img_name)
        
        # Load and display image
        pil_img = Image.open(img_path).convert("RGB")
        ax.imshow(pil_img)
        ax.axis('off')
        
        # Generate caption
        img_tensor = transform(pil_img).unsqueeze(0)
        generated_caption = generate_caption(encoder, decoder, img_tensor, vocab, max_len=20)
        
        # Get ground truth captions
        gt_captions = image2caps[img_name]
        
        # Format text
        title_text = f"Generated: {generated_caption}\n\n"
        title_text += "Ground Truth:\n"
        for i, cap in enumerate(gt_captions[:3], 1):  # Show first 3 GT captions
            title_text += f"  {i}. {cap}\n"
        
        ax.set_title(title_text, fontsize=10, loc='left', pad=10)
    
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Sample predictions saved to sample_predictions.png")

visualize_predictions(num_samples=5)

## 22. Generate Caption for Custom Image

In [ ]:
def caption_custom_image(image_path):
    """Generate caption for a custom image"""
    encoder.eval()
    decoder.eval()
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    # Load image
    pil_img = Image.open(image_path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0)
    
    # Generate caption
    caption = generate_caption(encoder, decoder, img_tensor, vocab, max_len=20)
    
    # Display
    plt.figure(figsize=(10, 8))
    plt.imshow(pil_img)
    plt.axis('off')
    plt.title(f"Generated Caption:\n{caption}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
    
    return caption

# Example usage:
# caption = caption_custom_image("path/to/your/image.jpg")
# print(f"Caption: {caption}")

## 23. Save and Load Model Functions

In [ ]:
def load_trained_model(checkpoint_path='best_model.pth'):
    """Load a trained model from checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=cfg.device)
    
    # Reconstruct vocabulary
    loaded_vocab = Vocabulary()
    loaded_vocab.itos = checkpoint['vocab']['itos']
    loaded_vocab.stoi = checkpoint['vocab']['stoi']
    
    # Reconstruct models
    loaded_encoder = EncoderCNN(
        encoder_dim=checkpoint['config']['encoder_dim']
    ).to(cfg.device)
    
    loaded_decoder = DecoderWithAttention(
        vocab_size=len(loaded_vocab),
        embed_dim=checkpoint['config']['embed_dim'],
        encoder_dim=checkpoint['config']['encoder_dim'],
        decoder_dim=checkpoint['config']['decoder_dim'],
        attention_dim=checkpoint['config']['attention_dim'],
        pad_idx=loaded_vocab.stoi[cfg.pad_token],
    ).to(cfg.device)
    
    # Load weights
    loaded_encoder.load_state_dict(checkpoint['encoder'])
    loaded_decoder.load_state_dict(checkpoint['decoder'])
    
    loaded_encoder.eval()
    loaded_decoder.eval()
    
    print(f"✓ Model loaded from {checkpoint_path}")
    return loaded_encoder, loaded_decoder, loaded_vocab

# Example usage:
# loaded_encoder, loaded_decoder, loaded_vocab = load_trained_model('best_model.pth')

## 24. Model Summary and GPU Memory Usage

In [ ]:
# Print model summary
print("\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)

# Count parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
encoder_trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
decoder_params = sum(p.numel() for p in decoder.parameters())
total_params = encoder_params + decoder_params
total_trainable = encoder_trainable + decoder_params

print(f"\nEncoder:")
print(f"  Total parameters: {encoder_params:,}")
print(f"  Trainable parameters: {encoder_trainable:,}")

print(f"\nDecoder:")
print(f"  Total parameters: {decoder_params:,}")
print(f"  Trainable parameters: {decoder_params:,}")

print(f"\nTotal:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {total_trainable:,}")

# GPU memory usage if using CUDA
if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

print("="*60)

## 25. Final Notes and Next Steps

### Training Complete! 🎉

**What we've accomplished:**
- ✅ Downloaded and prepared Flickr8k dataset
- ✅ Built vocabulary from training captions
- ✅ Implemented ResNet-50 CNN encoder
- ✅ Implemented LSTM decoder with Bahdanau attention
- ✅ Trained the model for multiple epochs
- ✅ Evaluated with BLEU-4 metric
- ✅ Generated sample captions

**Files saved:**
- `best_model.pth` - Best model checkpoint
- `training_curves.png` - Training/validation loss plots
- `sample_predictions.png` - Sample predictions

**Next steps:**
1. Try generating captions for your own images
2. Experiment with different hyperparameters
3. Implement beam search for better caption quality
4. Visualize attention weights on images
5. Train longer for better performance

**GPU Optimization Tips:**
- This notebook is optimized for GPU training
- Uses `non_blocking=True` for faster data transfer
- Uses `pin_memory=True` in DataLoaders
- Automatically detects CUDA/MPS/CPU

Enjoy exploring image captioning! 🚀